In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [3]:
unique_isin = tuple(isin['ISIN'])

In [4]:
isin['ISIN'].str[:2].unique()

array(['DE', 'IT', 'FR', 'ES'], dtype=object)

In [5]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [6]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [7]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [8]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

In [25]:
# Data prep
query = f"""

SELECT 
    s.lender_id AS dealer_id,
    s.lender_name AS dealer_name, 
    COUNT(*) as cnt
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER')
    AND s.security_isin IN {unique_isin}
GROUP BY s.lender_id, s.lender_name
ORDER BY s.lender_id, s.lender_name

"""

df_lending_d = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_25356\464336709.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending_d = pd.read_sql_query(query, cnxn)


In [26]:
# Data prep
query = f"""

SELECT 
    s.borrower_id AS dealer_id,
    s.borrower_name AS dealer_name, 
    COUNT(*) as cnt
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER')
    AND s.security_isin IN {unique_isin}
GROUP BY s.borrower_id, s.borrower_name
ORDER BY s.borrower_id, s.borrower_name
"""

df_borrowing_d = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_25356\2310026352.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing_d = pd.read_sql_query(query, cnxn)


In [27]:
df_d = pd.concat([df_lending_d, df_borrowing_d])[['dealer_id', 'dealer_name']].drop_duplicates().reset_index(drop=True)

In [28]:
df_borrowing_d['dealer_id'].unique()

array(['54930056FHWP7GIWYY08', '5493006QMFDDMYWIAM13',
       '549300FH0WJAPEHTIQ77', '549300ZK53CNGEEI6A29',
       '7LTWFZYICNSX8D621K86', 'DGQCSV2PHVF7I2743539',
       'K6Q0W1PS1L1O4IQL9C32', 'KX1WK48MPD4Y2NCUIZ63',
       'O2RNE8IBXP4R0TD8PU41', 'R0MUWSFPU8MPRO8K5P83',
       'RRAN7P32P0W0YY4XQW79', 'XKZZ2JZF41MRHTR1V493'], dtype=object)

In [29]:
df_d.to_excel('dealer_list.xlsx')

In [9]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id
ORDER BY s.business_date, s.borrower_id, s.lender_id

"""

df_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_23236\3176663820.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id
ORDER BY s.business_date, s.lender_id, s.borrower_id
"""

df_lending = pd.read_sql_query(query, cnxn)

In [11]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'fund_id', 'dealer_id'], how = 'outer')

In [12]:
df.to_csv('key dataframe\\fund_dealer_day.csv')

In [ ]:
cds_map = pd.read_excel('Data\\dealer_bloomberg.xlsx', sheet_name='Sheet1')
cds_wide = pd.read_excel('Data\\dealer_bloomberg.xlsx', sheet_name='Sheet3')

In [ ]:
cds_wide = cds_wide.loc[:, ~cds_wide.columns.str.endswith('.1')] # tickers shared by two LEIs were pulled twice
cds_long = cds_wide.melt(id_vars='Dates', var_name='Bloomberg', value_name='cds')
cds_long = cds_long.merge(cds_map[['dealer_id', 'Bloomberg']], on='Bloomberg', how='inner') # every LEI gets its parent's series
cds_long['period'] = cds_long['Dates'].dt.strftime('%Y-%m-%d')
cds_long = cds_long[['dealer_id', 'Bloomberg', 'period', 'cds']].rename(columns={'Bloomberg': 'bloomberg'})

In [ ]:
cds_long.to_csv('key dataframe\\dealer_cds.csv')